# Fase 5: Entrenamiento y comparación de modelos

En esta fase se entrenan múltiples modelos de clasificación y se comparan sus resultados mediante métricas adecuadas para seleccionar el mejor enfoque.

## Objetivo

Comparar distintos modelos de Machine Learning para identificar cuál tiene mejor desempeño en la predicción de inasistencia a citas médicas.

Se evaluarán:

- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- MLPClassifier

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
path = r"C:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\data\KaggleV2-May-2016-features.csv"

df = pd.read_csv(path)

print("Dataset con features cargado")
print(df.shape)
df.head()

Dataset con features cargado
(110526, 22)


,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,...,waiting_days,scheduled_weekday,appointment_weekday,is_same_day,age_group,has_comorbidity,risk_score,is_child_or_senior,appointment_month,schedule_hour
0,0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,...,0,4,4,1,senior,1,1,1,4,18
1,1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,...,0,4,4,1,adult,0,0,0,4,16
2,0,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,...,0,4,4,1,senior,0,0,1,4,16
3,0,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,...,0,4,4,1,child,0,0,1,4,17
4,0,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,...,0,4,4,1,adult,1,2,0,4,16


In [3]:
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'], errors='coerce')
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'], errors='coerce')

## Selección de variables

Se eliminan columnas que no aportan valor directo al modelo o que son redundantes.

In [4]:
# Target
y = df['No_show']

# Eliminamos columnas no necesarias
X = df.drop(columns=[
    'No_show',
    'ScheduledDay',
    'AppointmentDay'
])

In [5]:
categorical_cols = ['Neighbourhood', 'age_group']
numeric_cols = [col for col in X.columns if col not in categorical_cols]

print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)

Categorical: ['Neighbourhood', 'age_group']
Numeric: ['Gender', 'Age', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'waiting_days', 'scheduled_weekday', 'appointment_weekday', 'is_same_day', 'has_comorbidity', 'risk_score', 'is_child_or_senior', 'appointment_month', 'schedule_hour']


In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (88420, 19)
Test size: (22106, 19)


## Modelos a evaluar

In [8]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "MLPClassifier": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
}

In [9]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None

    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan,
        "conf_matrix": confusion_matrix(y_test, y_pred),
        "report": classification_report(y_test, y_pred)
    }

    return results

In [10]:
results_list = []

for name, model in models.items():
    print(f"\nEntrenando: {name}")

    results = evaluate_model(model, X_train, X_test, y_train, y_test)

    results_list.append({
        "model": name,
        "accuracy": results["accuracy"],
        "precision": results["precision"],
        "recall": results["recall"],
        "f1": results["f1"],
        "roc_auc": results["roc_auc"]
    })

    print(results["report"])
    print("Confusion Matrix:\n", results["conf_matrix"])


Entrenando: Logistic Regression
              precision    recall  f1-score   support

           0       0.80      1.00      0.89     17642
           1       0.45      0.01      0.02      4464

    accuracy                           0.80     22106
   macro avg       0.63      0.50      0.45     22106
weighted avg       0.73      0.80      0.71     22106

Confusion Matrix:
 [[17596    46]
 [ 4426    38]]

Entrenando: Decision Tree
              precision    recall  f1-score   support

           0       0.83      0.83      0.83     17642
           1       0.33      0.33      0.33      4464

    accuracy                           0.73     22106
   macro avg       0.58      0.58      0.58     22106
weighted avg       0.73      0.73      0.73     22106

Confusion Matrix:
 [[14728  2914]
 [ 3004  1460]]

Entrenando: Random Forest
              precision    recall  f1-score   support

           0       0.82      0.96      0.88     17642
           1       0.49      0.15      0.23      4

c:\Users\PC\Downloads\Recoleccion_Datos_PI\Recoleccion_Datos_PI\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


In [11]:
results_df = pd.DataFrame(results_list)

results_df = results_df.sort_values(by="f1", ascending=False)

results_df

,model,accuracy,precision,recall,f1,roc_auc
1,Decision Tree,0.732290,0.333791,0.327061,0.330391,0.582263
4,MLPClassifier,0.757939,0.361363,0.258961,0.301710,0.699334
2,Random Forest,0.797204,0.493141,0.153002,0.233544,0.741067
0,Logistic Regression,0.797702,0.452381,0.008513,0.016711,0.723446
3,Gradient Boosting,0.797973,0.470588,0.003584,0.007114,0.734285


## Interpretación de resultados

Se comparan los modelos utilizando métricas más allá de accuracy, especialmente:

- recall: importante para detectar pacientes que no asistirán
- f1-score: balance entre precisión y recall
- roc-auc: capacidad de discriminación del modelo

El mejor modelo se selecciona con base en estas métricas y no únicamente en la precisión global.

## Selección del modelo

Aunque algunos modelos presentaron una mayor precisión global (accuracy), se observó que estos modelos tenían un desempeño muy bajo en la detección de la clase minoritaria (pacientes que no asisten).

Dado que el objetivo principal del proyecto es identificar a los pacientes con mayor probabilidad de no asistir, se priorizaron métricas como recall y F1-score.

En este sentido, el modelo Decision Tree presentó el mejor equilibrio entre estas métricas, logrando un recall de 0.32 y un F1-score de 0.33, siendo el modelo más adecuado en esta fase del proyecto.

## Conclusión de la Fase 5

En esta fase se entrenaron múltiples modelos de Machine Learning y se compararon utilizando métricas relevantes.

Este enfoque permite justificar de manera sólida la selección del modelo final, evitando depender de un solo algoritmo.

Los resultados muestran que algunos modelos logran un mejor equilibrio entre precisión y recall, lo cual es clave para detectar pacientes con riesgo de inasistencia.